# Lab 5 — Predicting Garment Worker Productivity

Goal: predict how productive a garment factory team will be on a given day, and separately, predict whether a team will hit the productivity target that was set for it.

Two things we are predicting:

- **Regression target — `actual_productivity`**: a number between 0 and 1 (roughly, the fraction of the target output a team actually produced).
- **Classification target — `MeetsTarget`**: a yes/no label we create ourselves. It is 1 if `actual_productivity >= targeted_productivity` (the team met or beat its goal) and 0 otherwise.


In [31]:
import time
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None)


## 1. Loading the data



In [32]:
import os

df = pd.read_csv("garments_worker_productivity.csv")
print(f"Loaded dataset with shape {df.shape}")
df.head()


Loaded dataset with shape (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


## 2. Analysing the data


In [33]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

In [34]:
missing_counts = df.isna().sum()
missing_counts[missing_counts > 0]


wip    506
dtype: int64

Only one column has missing values: `wip` (work in progress), and it is missing for a large chunk of rows — roughly 40%. That is too much to just drop the rows, so later we will fill the gaps in with the median value instead of throwing away almost half the dataset.

The `department` column contains `"finishing"` and `"finishing "` (with a trailing space) as if they were two different departments. They are not — it is a typo in the raw data. We clean that up before doing anything else, otherwise the model would treat them as unrelated categories.


In [35]:
for col in ["quarter", "department", "day"]:
    print(col, "->", sorted(df[col].unique()))


quarter -> ['Quarter1', 'Quarter2', 'Quarter3', 'Quarter4', 'Quarter5']
department -> ['finishing', 'finishing ', 'sweing']
day -> ['Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [36]:
df["department"] = df["department"].str.strip().str.lower()
sorted(df["department"].unique())


['finishing', 'sweing']

Drop the `date` column. It is a raw calendar date (e.g. `1/1/2015`), and turning it into something a linear model can use meaningfully (a trend, a seasonal pattern) is a separate problem on its own. The `day` and `quarter` columns already tell the model which day of the week and which part of the month a row belongs to, so we do not lose much by dropping the raw date.

`team` is kept as a plain number. In reality a team number is more of a label than a quantity, but with only 12 teams and a dataset this size, treating it as one extra numeric column is a reasonable simplification and keeps the feature list manageable


In [37]:
df = df.drop(columns=["date"])

CATEGORICAL_COLS = ["quarter", "department", "day"]
NUMERIC_COLS = ["team", "targeted_productivity", "smv", "wip", "over_time",
                "incentive", "idle_time", "idle_men", "no_of_style_change", "no_of_workers"]

REGRESSION_TARGET = "actual_productivity"


## 3. Creating the classification target

`MeetsTarget` is 1 when the team's actual productivity was at least as high as what was targeted for it, and 0 otherwise.


In [38]:
df["MeetsTarget"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)
df["MeetsTarget"].value_counts(normalize=True).rename("share of rows")


MeetsTarget
1    0.730994
0    0.269006
Name: share of rows, dtype: float64

About 73% of teams meet or beat their target and 27% fall short. It is an imbalanced label, so besides accuracy we also look at precision, recall and F1 — accuracy alone can look good on an imbalanced dataset just by always guessing the majority class.


## 4. train/test split


In [39]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)

train_df = df.loc[train_idx].reset_index(drop=True)
test_df = df.loc[test_idx].reset_index(drop=True)

print(f"Training rows: {len(train_df)}   Test rows: {len(test_df)}")


Training rows: 957   Test rows: 240


# Part A — Scikit-learn implementation



###  Preprocessing pipeline

For the numeric columns we fill missing values with the median and then standardize (subtract the mean, divide by the standard deviation) so that columns measured in very different units — minutes, worker counts, overtime minutes — don't dominate the model just because their raw numbers are bigger.

For the categorical columns (`quarter`, `department`, `day`) we one-hot encode: each category becomes its own 0/1 column. We drop one category per column (`drop='first'`) to avoid redundant columns that are perfectly predictable from the others.

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, precision_score, recall_score, f1_score)

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
])

def make_preprocessor():
    # a fresh, un-fitted copy each time we build a model, so the regression
    # and classification pipelines never accidentally share fitted state
    return ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_COLS),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
    ])


###  Regression: predicting `actual_productivity`


In [41]:
feature_cols = NUMERIC_COLS + CATEGORICAL_COLS

X_train_reg = train_df[feature_cols]
y_train_reg = train_df[REGRESSION_TARGET]
X_test_reg = test_df[feature_cols]
y_test_reg = test_df[REGRESSION_TARGET]

reg_preprocessor = make_preprocessor()

t0 = time.perf_counter()
X_train_reg_t = reg_preprocessor.fit_transform(X_train_reg)
sk_linreg = LinearRegression()
sk_linreg.fit(X_train_reg_t, y_train_reg)
sk_reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
X_test_reg_t = reg_preprocessor.transform(X_test_reg)
sk_reg_pred = sk_linreg.predict(X_test_reg_t)
sk_reg_predict_time = time.perf_counter() - t0

sk_reg_metrics = {
    "MAE": mean_absolute_error(y_test_reg, sk_reg_pred),
    "RMSE": mean_squared_error(y_test_reg, sk_reg_pred) ** 0.5,
    "R2": r2_score(y_test_reg, sk_reg_pred),
    "train_time_sec": sk_reg_train_time,
    "predict_time_sec": sk_reg_predict_time,
}
sk_reg_metrics


{'MAE': 0.10877143579760805,
 'RMSE': 0.14813367474432454,
 'R2': 0.1735763766527343,
 'train_time_sec': 0.019232999999076128,
 'predict_time_sec': 0.007373099913820624}

###  Classification: predicting `MeetsTarget`


In [42]:
CLASSIFICATION_TARGET = "MeetsTarget"

X_train_clf = train_df[feature_cols]
y_train_clf = train_df[CLASSIFICATION_TARGET]
X_test_clf = test_df[feature_cols]
y_test_clf = test_df[CLASSIFICATION_TARGET]

clf_preprocessor = make_preprocessor()

t0 = time.perf_counter()
X_train_clf_t = clf_preprocessor.fit_transform(X_train_clf)
sk_logreg = LogisticRegression(max_iter=1000)
sk_logreg.fit(X_train_clf_t, y_train_clf)
sk_clf_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
X_test_clf_t = clf_preprocessor.transform(X_test_clf)
sk_clf_pred = sk_logreg.predict(X_test_clf_t)
sk_clf_predict_time = time.perf_counter() - t0

sk_clf_metrics = {
    "Accuracy": accuracy_score(y_test_clf, sk_clf_pred),
    "Precision": precision_score(y_test_clf, sk_clf_pred, zero_division=0),
    "Recall": recall_score(y_test_clf, sk_clf_pred, zero_division=0),
    "F1": f1_score(y_test_clf, sk_clf_pred, zero_division=0),
    "train_time_sec": sk_clf_train_time,
    "predict_time_sec": sk_clf_predict_time,
}
sk_clf_metrics


{'Accuracy': 0.7416666666666667,
 'Precision': 0.7625570776255708,
 'Recall': 0.943502824858757,
 'F1': 0.8434343434343434,
 'train_time_sec': 0.03407280007377267,
 'predict_time_sec': 0.005851400084793568}

# Part B — Only NumPy and Pandas




### Filling in missing values

Learn the median from the training data only, remember it, and use that same number to fill gaps in both the training and the test data.


In [43]:
class ManualMedianImputer:
    """Fills missing numeric values with the median learned from the training data."""

    def fit(self, frame, columns):
        self.fill_values_ = {col: frame[col].median() for col in columns}
        return self

    def transform(self, frame, columns):
        frame = frame.copy()
        for col in columns:
            frame[col] = frame[col].fillna(self.fill_values_[col])
        return frame


### One-hot encoding

In [44]:
class ManualOneHotEncoder:
    """One-hot encodes categorical columns, dropping the first category of each."""

    def __init__(self, drop_first=True):
        self.drop_first = drop_first

    def fit(self, frame, columns):
        self.categories_ = {col: sorted(frame[col].unique()) for col in columns}
        return self

    def transform(self, frame, columns):
        encoded_columns = []
        for col in columns:
            categories = self.categories_[col]
            categories_to_use = categories[1:] if self.drop_first else categories
            for category in categories_to_use:
                encoded_columns.append((frame[col] == category).astype(float).rename(f"{col}__{category}"))
        if not encoded_columns:
            return pd.DataFrame(index=frame.index)
        return pd.concat(encoded_columns, axis=1)


### 11. Standard scaling



In [45]:
class ManualStandardScaler:
    """Standardizes numeric columns: (x - mean) / std, using training statistics."""

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0, ddof=0)
        self.std_[self.std_ == 0] = 1.0  # a constant column would otherwise divide by zero
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_


In [46]:
def manual_preprocess(train_frame, test_frame, numeric_cols, categorical_cols):
    """Fits every step on train_frame only, then applies the same fitted steps to both frames."""

    imputer = ManualMedianImputer().fit(train_frame, numeric_cols)
    train_filled = imputer.transform(train_frame, numeric_cols)
    test_filled = imputer.transform(test_frame, numeric_cols)

    encoder = ManualOneHotEncoder(drop_first=True).fit(train_frame, categorical_cols)
    train_encoded = encoder.transform(train_filled, categorical_cols)
    test_encoded = encoder.transform(test_filled, categorical_cols)

    scaler = ManualStandardScaler().fit(train_filled[numeric_cols].values.astype(float))
    train_scaled = scaler.transform(train_filled[numeric_cols].values.astype(float))
    test_scaled = scaler.transform(test_filled[numeric_cols].values.astype(float))

    X_train = np.hstack([train_scaled, train_encoded.values])
    X_test = np.hstack([test_scaled, test_encoded.values])
    return X_train, X_test


### Linear Regression (closed-form / normal equation)




In [47]:
def add_intercept_column(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])


class ManualLinearRegression:
    """Closed-form (ridge) linear regression: theta = (X'X + l2*I)^-1 X'y."""

    def __init__(self, l2=0.0):
        self.l2 = l2

    def fit(self, X, y):
        X_with_intercept = add_intercept_column(X)
        n_features = X_with_intercept.shape[1]
        penalty = self.l2 * np.eye(n_features)
        penalty[0, 0] = 0.0  # never penalize the intercept
        y_col = np.asarray(y).reshape(-1, 1)
        self.theta_ = np.linalg.pinv(X_with_intercept.T @ X_with_intercept + penalty) @ X_with_intercept.T @ y_col
        return self

    def predict(self, X):
        X_with_intercept = add_intercept_column(X)
        return (X_with_intercept @ self.theta_).ravel()


### 13. Logistic Regression (gradient descent)



In [48]:
class ManualLogisticRegression:
    """Logistic regression trained with vectorized batch gradient descent."""

    def __init__(self, learning_rate=0.1, n_iterations=3000, l2=0.0, tol=1e-7, check_every=20):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.l2 = l2
        self.tol = tol
        self.check_every = check_every

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)  # avoids overflow warnings for very large/small scores
        return 1.0 / (1.0 + np.exp(-z))

    def _loss(self, y, p):
        eps = 1e-12
        return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

    def fit(self, X, y):
        X_with_intercept = add_intercept_column(X)
        n_samples, n_features = X_with_intercept.shape
        theta = np.zeros((n_features, 1))
        y_col = np.asarray(y).reshape(-1, 1).astype(float)

        previous_loss = np.inf
        for iteration in range(self.n_iterations):
            probabilities = self._sigmoid(X_with_intercept @ theta)
            gradient = X_with_intercept.T @ (probabilities - y_col) / n_samples
            l2_term = (self.l2 / n_samples) * theta
            l2_term[0] = 0.0  # never penalize the intercept
            theta -= self.learning_rate * (gradient + l2_term)

            if iteration % self.check_every == 0:
                current_loss = self._loss(y_col, probabilities)
                if abs(previous_loss - current_loss) < self.tol:
                    break
                previous_loss = current_loss

        self.theta_ = theta
        self.n_iterations_run_ = iteration + 1
        return self

    def predict_proba(self, X):
        X_with_intercept = add_intercept_column(X)
        return self._sigmoid(X_with_intercept @ self.theta_).ravel()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [49]:
def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def manual_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def manual_r2(y_true, y_pred):
    ss_residual = np.sum((y_true - y_pred) ** 2)
    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_residual / ss_total

def manual_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def manual_precision(y_true, y_pred):
    true_positive = np.sum((y_pred == 1) & (y_true == 1))
    false_positive = np.sum((y_pred == 1) & (y_true == 0))
    return true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0.0

def manual_recall(y_true, y_pred):
    true_positive = np.sum((y_pred == 1) & (y_true == 1))
    false_negative = np.sum((y_pred == 0) & (y_true == 1))
    return true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0.0

def manual_f1(precision, recall):
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0


In [50]:
X_train_reg_manual, X_test_reg_manual = manual_preprocess(
    train_df[feature_cols], test_df[feature_cols], NUMERIC_COLS, CATEGORICAL_COLS
)

t0 = time.perf_counter()
manual_linreg = ManualLinearRegression(l2=1.0).fit(X_train_reg_manual, y_train_reg)
manual_reg_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
manual_reg_pred = manual_linreg.predict(X_test_reg_manual)
manual_reg_predict_time = time.perf_counter() - t0

y_test_reg_arr = y_test_reg.values
manual_reg_metrics = {
    "MAE": manual_mae(y_test_reg_arr, manual_reg_pred),
    "RMSE": manual_rmse(y_test_reg_arr, manual_reg_pred),
    "R2": manual_r2(y_test_reg_arr, manual_reg_pred),
    "train_time_sec": manual_reg_train_time,
    "predict_time_sec": manual_reg_predict_time,
}
manual_reg_metrics


{'MAE': np.float64(0.10861211493231408),
 'RMSE': np.float64(0.14800687354497902),
 'R2': np.float64(0.17499059467517386),
 'train_time_sec': 0.0007780001033097506,
 'predict_time_sec': 8.25999304652214e-05}

In [51]:
X_train_clf_manual, X_test_clf_manual = manual_preprocess(
    train_df[feature_cols], test_df[feature_cols], NUMERIC_COLS, CATEGORICAL_COLS
)

t0 = time.perf_counter()
manual_logreg = ManualLogisticRegression(learning_rate=0.5, n_iterations=3000, l2=0.5).fit(
    X_train_clf_manual, y_train_clf
)
manual_clf_train_time = time.perf_counter() - t0
print(f"gradient descent stopped after {manual_logreg.n_iterations_run_} iterations")

t0 = time.perf_counter()
manual_clf_pred = manual_logreg.predict(X_test_clf_manual)
manual_clf_predict_time = time.perf_counter() - t0

y_test_clf_arr = y_test_clf.values
manual_precision_val = manual_precision(y_test_clf_arr, manual_clf_pred)
manual_recall_val = manual_recall(y_test_clf_arr, manual_clf_pred)
manual_clf_metrics = {
    "Accuracy": manual_accuracy(y_test_clf_arr, manual_clf_pred),
    "Precision": manual_precision_val,
    "Recall": manual_recall_val,
    "F1": manual_f1(manual_precision_val, manual_recall_val),
    "train_time_sec": manual_clf_train_time,
    "predict_time_sec": manual_clf_predict_time,
}
manual_clf_metrics


gradient descent stopped after 3000 iterations


{'Accuracy': np.float64(0.7541666666666667),
 'Precision': np.float64(0.7731481481481481),
 'Recall': np.float64(0.943502824858757),
 'F1': np.float64(0.8498727735368956),
 'train_time_sec': 0.1111339998897165,
 'predict_time_sec': 0.00018090009689331055}

# Part C — Comparing the two versions, then improving the manual one


In [52]:
regression_comparison = pd.DataFrame({
    "Scikit-learn": sk_reg_metrics,
    "Manual (NumPy/Pandas)": manual_reg_metrics,
}).T
regression_comparison


,MAE,RMSE,R2,train_time_sec,predict_time_sec
Scikit-learn,0.108771,0.148134,0.173576,0.019233,0.007373
Manual (NumPy/Pandas),0.108612,0.148007,0.174991,0.000778,0.000083


In [53]:
classification_comparison = pd.DataFrame({
    "Scikit-learn": sk_clf_metrics,
    "Manual (NumPy/Pandas)": manual_clf_metrics,
}).T
classification_comparison


,Accuracy,Precision,Recall,F1,train_time_sec,predict_time_sec
Scikit-learn,0.741667,0.762557,0.943503,0.843434,0.034073,0.005851
Manual (NumPy/Pandas),0.754167,0.773148,0.943503,0.849873,0.111134,0.000181


The real difference so far shows up in **training time**: gradient descent for logistic regression needs a lot of small steps, so it is noticeably slower to train than Scikit-learn's more efficient internal solver, even though both reach a similar answer.


### Optimizing Logistic Regression

This method uses the *curvature* of the loss — the Hessian matrix $H$ — to jump much closer to the answer on every step:

$$H = X^T \, \text{diag}\big(p(1-p)\big) \, X \qquad \theta \leftarrow \theta - H^{-1}\nabla L(\theta)$$

Here $p$ is the vector of current predicted probabilities, and $p(1-p)$ measures how uncertain each prediction currently is. Computing $H$ costs more per step than a gradient-descent step, but far fewer steps are needed overall — typically single digits to a few dozen instead of thousands.


In [54]:
class ManualLogisticRegressionNewton:

    def __init__(self, n_iterations=25, l2=0.5, tol=1e-8):
        self.n_iterations = n_iterations
        self.l2 = l2
        self.tol = tol

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        X_with_intercept = add_intercept_column(X)
        n_samples, n_features = X_with_intercept.shape
        theta = np.zeros((n_features, 1))
        y_col = np.asarray(y).reshape(-1, 1).astype(float)

        penalty = self.l2 * np.eye(n_features)
        penalty[0, 0] = 0.0
        previous_loss = np.inf

        for iteration in range(self.n_iterations):
            probabilities = self._sigmoid(X_with_intercept @ theta)
            gradient = X_with_intercept.T @ (probabilities - y_col) + penalty @ theta
            weight = (probabilities * (1 - probabilities)).ravel()
            hessian = X_with_intercept.T @ (X_with_intercept * weight[:, None]) + penalty + 1e-8 * np.eye(n_features)
            theta -= np.linalg.solve(hessian, gradient)

            eps = 1e-12
            loss = -np.sum(y_col * np.log(probabilities + eps) + (1 - y_col) * np.log(1 - probabilities + eps))
            loss += 0.5 * self.l2 * np.sum(theta[1:] ** 2)
            if abs(previous_loss - loss) < self.tol:
                break
            previous_loss = loss

        self.theta_ = theta
        self.n_iterations_run_ = iteration + 1
        return self

    def predict_proba(self, X):
        X_with_intercept = add_intercept_column(X)
        return self._sigmoid(X_with_intercept @ self.theta_).ravel()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [55]:
t0 = time.perf_counter()
manual_logreg_v2 = ManualLogisticRegressionNewton(n_iterations=25, l2=0.5).fit(X_train_clf_manual, y_train_clf)
manual_clf_train_time_v2 = time.perf_counter() - t0
print(f"Newton's method stopped after {manual_logreg_v2.n_iterations_run_} iterations")

t0 = time.perf_counter()
manual_clf_pred_v2 = manual_logreg_v2.predict(X_test_clf_manual)
manual_clf_predict_time_v2 = time.perf_counter() - t0

precision_v2 = manual_precision(y_test_clf_arr, manual_clf_pred_v2)
recall_v2 = manual_recall(y_test_clf_arr, manual_clf_pred_v2)
manual_clf_metrics_v2 = {
    "Accuracy": manual_accuracy(y_test_clf_arr, manual_clf_pred_v2),
    "Precision": precision_v2,
    "Recall": recall_v2,
    "F1": manual_f1(precision_v2, recall_v2),
    "train_time_sec": manual_clf_train_time_v2,
    "predict_time_sec": manual_clf_predict_time_v2,
}
manual_clf_metrics_v2


Newton's method stopped after 9 iterations


{'Accuracy': np.float64(0.7541666666666667),
 'Precision': np.float64(0.7731481481481481),
 'Recall': np.float64(0.943502824858757),
 'F1': np.float64(0.8498727735368956),
 'train_time_sec': 0.006859299959614873,
 'predict_time_sec': 0.0006287000142037868}

### A small feature-engineering fix



In [56]:
train_df_fe = train_df.copy()
test_df_fe = test_df.copy()
for frame in (train_df_fe, test_df_fe):
    frame["targeted_x_incentive"] = frame["targeted_productivity"] * frame["incentive"]
    frame["targeted_productivity_sq"] = frame["targeted_productivity"] ** 2

NUMERIC_COLS_EXTENDED = NUMERIC_COLS + ["targeted_x_incentive", "targeted_productivity_sq"]
extended_feature_cols = NUMERIC_COLS_EXTENDED + CATEGORICAL_COLS

# build the extended version alongside the original (not overwriting it yet) so we can compare fairly
X_train_reg_extended, X_test_reg_extended = manual_preprocess(
    train_df_fe[extended_feature_cols], test_df_fe[extended_feature_cols],
    NUMERIC_COLS_EXTENDED, CATEGORICAL_COLS
)

# fit both versions on the same train/validation split, with the same ridge strength, so the
# only thing that differs between them is whether the two extra columns are present
check_train_pos, check_val_pos = train_test_split(
    np.arange(len(train_df)), test_size=0.2, random_state=RANDOM_STATE
)
y_check_train = y_train_reg.iloc[check_train_pos]
y_check_val = y_train_reg.iloc[check_val_pos].values

model_original_cols = ManualLinearRegression(l2=1.0).fit(
    X_train_reg_manual[check_train_pos], y_check_train
)
rmse_original = manual_rmse(y_check_val, model_original_cols.predict(X_train_reg_manual[check_val_pos]))

model_extended_cols = ManualLinearRegression(l2=1.0).fit(
    X_train_reg_extended[check_train_pos], y_check_train
)
rmse_extended = manual_rmse(y_check_val, model_extended_cols.predict(X_train_reg_extended[check_val_pos]))

print(f"validation RMSE, original columns:        {rmse_original:.4f}")
print(f"validation RMSE, with the 2 extra columns: {rmse_extended:.4f}")

# the extra columns genuinely help, so we keep them from here on
X_train_reg_manual, X_test_reg_manual = X_train_reg_extended, X_test_reg_extended


validation RMSE, original columns:        0.1641
validation RMSE, with the 2 extra columns: 0.1572


### Tuning the ridge strength for Linear Regression

We also try to improve the regression model by searching over a few values of $\lambda$.


In [57]:
train_positions, val_positions = train_test_split(
    np.arange(len(train_df)), test_size=0.2, random_state=RANDOM_STATE
)
X_fit_only, X_val_only = X_train_reg_manual[train_positions], X_train_reg_manual[val_positions]
y_fit_only, y_val_only = y_train_reg.iloc[train_positions], y_train_reg.iloc[val_positions]

candidate_l2_values = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
validation_scores = []
for l2 in candidate_l2_values:
    candidate_model = ManualLinearRegression(l2=l2).fit(X_fit_only, y_fit_only)
    val_pred = candidate_model.predict(X_val_only)
    validation_scores.append(manual_rmse(y_val_only.values, val_pred))

pd.DataFrame({"l2": candidate_l2_values, "validation_RMSE": validation_scores})


,l2,validation_RMSE
0,0.0,0.162974
1,0.1,0.161418
2,0.5,0.158571
3,1.0,0.157195
4,2.0,0.156030
5,5.0,0.155047
6,10.0,0.154788
7,20.0,0.154801


In [58]:
best_l2 = candidate_l2_values[int(np.argmin(validation_scores))]
print(f"Best l2 found on the validation slice: {best_l2}")

# refit on the FULL training set with the chosen l2, then score once on the untouched test set
t0 = time.perf_counter()
manual_linreg_v2 = ManualLinearRegression(l2=best_l2).fit(X_train_reg_manual, y_train_reg)
manual_reg_train_time_v2 = time.perf_counter() - t0

t0 = time.perf_counter()
manual_reg_pred_v2 = manual_linreg_v2.predict(X_test_reg_manual)
manual_reg_predict_time_v2 = time.perf_counter() - t0

manual_reg_metrics_v2 = {
    "MAE": manual_mae(y_test_reg_arr, manual_reg_pred_v2),
    "RMSE": manual_rmse(y_test_reg_arr, manual_reg_pred_v2),
    "R2": manual_r2(y_test_reg_arr, manual_reg_pred_v2),
    "train_time_sec": manual_reg_train_time_v2,
    "predict_time_sec": manual_reg_predict_time_v2,
}
manual_reg_metrics_v2


Best l2 found on the validation slice: 10.0


{'MAE': np.float64(0.10637382880859984),
 'RMSE': np.float64(0.14586051928840554),
 'R2': np.float64(0.1987452062821592),
 'train_time_sec': 0.0027220998890697956,
 'predict_time_sec': 0.00029710005037486553}

### Final comparison


In [59]:
final_regression_comparison = pd.DataFrame({
    "Scikit-learn": sk_reg_metrics,
    "Manual (baseline)": manual_reg_metrics,
    "Manual (optimized)": manual_reg_metrics_v2,
}).T
final_regression_comparison


,MAE,RMSE,R2,train_time_sec,predict_time_sec
Scikit-learn,0.108771,0.148134,0.173576,0.019233,0.007373
Manual (baseline),0.108612,0.148007,0.174991,0.000778,0.000083
Manual (optimized),0.106374,0.145861,0.198745,0.002722,0.000297


In [60]:
final_classification_comparison = pd.DataFrame({
    "Scikit-learn": sk_clf_metrics,
    "Manual (baseline)": manual_clf_metrics,
    "Manual (optimized)": manual_clf_metrics_v2,
}).T
final_classification_comparison


,Accuracy,Precision,Recall,F1,train_time_sec,predict_time_sec
Scikit-learn,0.741667,0.762557,0.943503,0.843434,0.034073,0.005851
Manual (baseline),0.754167,0.773148,0.943503,0.849873,0.111134,0.000181
Manual (optimized),0.754167,0.773148,0.943503,0.849873,0.006859,0.000629
